# Seminário de PLN - Tema 2: Detector de Sentimentos Baseado em Léxico

### **Enunciado**

Classificar textos como positivos, negativos ou neutros. Calcular percentuais de cada classe. Considerar intensificadores como: muito bom, extremamente ruim.

**O que é Tokenizar?**

Em Processamento de Linguagem Natural (NLP), é o processo de dividir textos grandes em unidades menores (palavras ou caracteres) para que algoritmos de inteligência artificial possam entender e analisar a linguagem humana.

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
import requests
from collections import Counter

# Baixa os pacotes de tokenização do NLTK para a língua portuguesa
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Definição dos Léxicos
positive_lexicon = {
    "excelente", "ótimo", "excelente", "ótimo",
    "bom", "rápido", "eficiente", "gostei",
}

negative_lexicon = {
    "ruim", "péssimo", "lento", "horrível", "problema", "falha",
}

# Intensificadores
intensifiers = {
    "muito", "extremamente", "super", "bastante", "bem",
}

In [ ]:
# Leitura dos comentários no arquivo .txt
def read_comments():
  URL = ("https://raw.githubusercontent.com/pv-Chagas/"
  "uema-actv-lexicon-based-sentiment-analysis/main/docs/comentarios.txt")    # arquivo remoto
  local_path = "docs/comentarios.txt"                                        # arquivo local

  try:
    with open(local_path, "r", encoding="utf-8") as f:
      content = [line.strip() for line in f if line.strip()]
  except FileNotFoundError:
    text = requests.get(URL).text
    content = [line.strip() for line in text.splitlines() if line.strip()]
  
  return content

# Tokenização: Pega os comentários e converte em tokens
def tokenize(text: str):
  cleaned_text = text.lower() # letras minúsculas
  cleaned_text = re.sub(r"[^\w\s]", "", cleaned_text) # remove pontuações
  tokens = word_tokenize(cleaned_text, language="portuguese")
  return tokens

# Análise de sentimento: Verifica quais tokens estão nos léxicos
def analyze_sentiment(tokens):
  positive_count = 0
  negative_count = 0
  positive_score = 0
  negative_score = 0

  for i, token in enumerate(tokens):
    has_intensifier = i > 0 and tokens[i-1] in intensifiers
    weight = 2 if has_intensifier else 1

    if token in positive_lexicon:
      positive_count += 1
      positive_score += weight

    elif token in negative_lexicon:
      negative_count += 1
      negative_score += weight

  score = positive_score - negative_score

  if score > 0: sentiment_label = "POSITIVO"
  elif score < 0: sentiment_label = "NEGATIVO"
  else: sentiment_label = "NEUTRO"

  return {
      "positive": positive_count,
      "negative": negative_count,
      "label": sentiment_label,
  }

# Execução do NLP: Rotina principal
def run_nlp_pipeline():
  comments = read_comments()
  total_comments = len(comments)
  class_counts = Counter()

  if total_comments == 0:
    print("Nenhum comentário no arquivo.")
    return

  print("----- ANÁLISE DE CADA COMENTÁRIO -----\n")
  for comment in comments:
    tokens = tokenize(comment)
    result = analyze_sentiment(tokens)
    class_counts[result["label"]] += 1

    print(f"Comentário: {comment}")
    print(f"Positivas: {result['positive']} Negativas: {result['negative']}.", end="\t")
    print(f"Classificação: {result['label']}\n")

  print("----- RESUMO E PERCENTUAIS POR CLASSE -----")
  for label in ("POSITIVO", "NEGATIVO", "NEUTRO"):
    count = class_counts[label]
    percentage = (count / total_comments) * 100
    print(f"{label}: {count} de {total_comments} ({percentage:.2f}%)")


In [ ]:
if __name__ == "__main__":
  run_nlp_pipeline()

----- ANÁLISE DE CADA COMENTÁRIO -----

Comentário: O sistema é excelente e muito rápido.
Positivas: 2 Negativas: 0.	Classificação: POSITIVO

Comentário: O atendimento foi horrível.
Positivas: 0 Negativas: 1.	Classificação: NEGATIVO

Comentário: O produto funciona normalmente.
Positivas: 0 Negativas: 0.	Classificação: NEUTRO

----- RESUMO E PERCENTUAIS POR CLASSE -----
POSITIVO: 1 de 3 (33.33%)
NEGATIVO: 1 de 3 (33.33%)
NEUTRO: 1 de 3 (33.33%)
